In [91]:
import os
import pandas as pd
from datetime import datetime
import importlib

In [92]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [93]:
compact = False

In [94]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 1:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading 2024-08-30_23-46-12.parquet
Reading 2024-08-30_22-21-59.parquet
Reading 2024-08-30_23-08-29.parquet
Reading 2024-08-30_23-34-09.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
5967921264,NaN,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Alun,HumanPlayer,call,2,0.4707,0.014121,0,"[5, 3, 0, 0, 0]"
4994597600,5.967921e+09,[],"[96, 89]",0,"[4, 11]","[4, 11]","[True, True]","[False, False]",0,4,Alun,HumanPlayer,fold,0,0.4707,0.035302,0,"[5, 3, 0, 0, 0]"
6073122528,NaN,[],"[92, 100]",0,"[4, 4]","[4, 4]","[False, True]","[False, False]",1,4,Alun,HumanPlayer,check,0,0.5593,0.022372,0,"[10, 4, 0, 0, 0]"
6073122336,6.073123e+09,"[50, 37, 12]","[92, 100]",0,"[0, 0]","[4, 4]","[False, True]","[False, False]",1,4,Alun,HumanPlayer,check,0,0.6109,0.024436,1,"[11, 12, 10, 4, 0]"
4992440544,6.073122e+09,"[50, 37, 12, 42]","[92, 100]",0,"[0, 0]","[4, 4]","[False, True]","[False, False]",1,4,Alun,HumanPlayer,check,0,0.5616,0.022464,1,"[11, 12, 10, 4, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6106980208,6.106977e+09,"[32, 10, 38]","[170, 22]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.9373,0.037492,3,"[12, 12, 12, 0, 0]"
6107145264,6.106980e+09,"[32, 10, 38]","[170, 16]",0,"[0, 6]","[4, 10]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.9373,0.065611,3,"[12, 12, 12, 0, 0]"
6107099680,6.107145e+09,"[32, 10, 38, 45]","[164, 16]",0,"[0, 0]","[10, 10]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.9965,0.099650,6,"[12, 6, 0, 0, 0]"


In [95]:
raw_df.dtypes

prev_entry           float64
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

## Process data

In [96]:
df = Processor(raw_df).get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name
5967921264,28de98a9-1719-4641-a0ea-2377417ddcb4,0,0,0,0,0,0,0,0,0,...,0,0,0,call,2,0,0.4707,0.014121,preflop,Alun
4994597600,28de98a9-1719-4641-a0ea-2377417ddcb4,0,0,0,0,0,2,0,0,0,...,0,0,0,fold,0,0,0.4707,0.035302,preflop,Alun
6073122528,ec7e8a9d-824e-4f8a-b246-b2269e5529ed,0,0,0,0,0,0,0,0,0,...,0,0,0,check,0,0,0.5593,0.022372,preflop,Alun
6073122336,ec7e8a9d-824e-4f8a-b246-b2269e5529ed,0,0,0,0,0,0,0,0,0,...,0,0,0,check,0,0,0.6109,0.024436,flop,Alun
4992440544,ec7e8a9d-824e-4f8a-b246-b2269e5529ed,0,0,0,0,0,0,0,0,0,...,0,0,0,check,0,0,0.5616,0.022464,turn,Alun
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6106980208,daec6871-ada1-411d-af14-16516ecd0e7a,0,0,0,0,0,2,0,0,0,...,0,0,0,check,0,3,0.9373,0.037492,flop,Tord
6107145264,daec6871-ada1-411d-af14-16516ecd0e7a,0,0,0,0,0,2,0,0,0,...,0,0,0,call,6,3,0.9373,0.065611,flop,Tord
6107099680,daec6871-ada1-411d-af14-16516ecd0e7a,0,0,0,0,0,2,6,0,0,...,1,0,0,check,0,5,0.9965,0.099650,turn,Tord
6107155472,daec6871-ada1-411d-af14-16516ecd0e7a,0,0,0,0,0,2,6,0,0,...,1,0,0,call,10,5,0.9965,0.149475,turn,Tord


In [97]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [98]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [99]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [100]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_call_showdown,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name
5967921264,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,call,2,preflop,Alun
4994597600,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,0,fold,0,preflop,Alun
6073122528,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,check,0,preflop,Alun
6073122336,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,check,0,flop,Alun
4992440544,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,check,0,turn,Alun
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6106980208,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,check,0,flop,Tord
6107145264,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,flop,Tord
6107099680,0,0,0,0,0,2,6,0,0,0,...,0,0,1,1,0,0,check,0,turn,Tord
6107155472,0,0,0,0,0,2,6,0,0,0,...,0,0,1,1,0,0,call,10,turn,Tord


In [101]:
y

5967921264    0
4994597600    0
6073122528    0
6073122336    0
4992440544    0
             ..
6106980208    3
6107145264    3
6107099680    5
6107155472    5
6107143664    5
Name: excess_rank, Length: 554, dtype: int64

In [102]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=1000
            ),
        ),
    ]
)

In [103]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (459, 34)
Test shape: (95, 34)


In [104]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.8315789473684211


In [105]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.8315789473684211
Mean goodness:  0.7641974610659189


,0,1,2,3,4,5,true,pred,correct,goodness
0,0.956255,0.040853,0.000449,0.001350,0.000245,8.469917e-04,0,0,True,0.956255
1,0.998710,0.001278,0.000002,0.000008,0.000001,4.592460e-07,0,0,True,0.998710
2,0.984399,0.014160,0.000134,0.000850,0.000079,3.775381e-04,0,0,True,0.984399
3,0.819876,0.168337,0.001246,0.007828,0.000689,2.023978e-03,1,0,False,0.168337
4,0.248791,0.694061,0.020840,0.005236,0.006483,2.458826e-02,1,1,True,0.694061
...,...,...,...,...,...,...,...,...,...,...
90,0.916160,0.083171,0.000160,0.000418,0.000025,6.618531e-05,0,0,True,0.916160
91,0.984290,0.008640,0.000002,0.006949,0.000105,1.400027e-05,0,0,True,0.984290
92,0.975543,0.021516,0.000119,0.001783,0.000063,9.768234e-04,0,0,True,0.975543
93,0.944321,0.044513,0.000590,0.008591,0.000279,1.705497e-03,0,0,True,0.944321
